# Module 0.2: Git Version Control for Scientists

---

### Learning Objectives

By the end of this module, you will be able to:
- Explain why version control is essential for reproducible research
- Initialize a Git repository and track changes
- Navigate project history (log, diff, show)
- Use branches to develop features and experiments in isolation
- Collaborate with others using GitHub (push, pull, pull requests)
- Undo mistakes safely
- Write a proper `.gitignore` for bioinformatics projects

**Prerequisites:** Basic command-line familiarity (Module 0.1)

**Estimated time:** 2-3 hours

---

## Why this notebook matters

Version control is what separates reproducible science from "works on my machine." Without Git, a six-month-old analysis is a mystery: which parameters did you use? Which version of the script? When a reviewer asks you to re-run with different thresholds, you need to find the exact code. Git also enables collaboration without overwriting each other's work, and makes it easy to publish your analysis code alongside your paper.


## How to use this notebook
1. Run cells top-to-bottom — each section builds a Git repository that later sections continue to use.
2. The `%%bash` cells run real git commands. Read the git output carefully — it tells you exactly what happened.
3. After each commit, run `git log --oneline` to see the growing history.
4. To experiment with undoing things, create a separate directory so you can make mistakes without worrying.


## Common stumbling points

- **Staging vs. committing**: `git add` does not save your work permanently — it just marks files for the *next* commit. Files in the working directory that are not staged will not be included in the commit.
- **Commit messages are forever**: Write messages that explain *why* you made the change, not just *what* changed. "Fix bug" is useless six months later; "Fix off-by-one error in exon boundary parsing" is useful.
- **Never commit large data files**: Git is not a data store. A single BAM file committed to a repo can make it permanently bloated and impossible to clone quickly.
- **`git reset --hard` is destructive**: Unlike most Git operations, `--hard` discards your changes with no undo. Use `--soft` to undo commits while keeping your edits.
- **`HEAD~1` means "one commit before the current one"**: `git diff HEAD~1` shows what changed in the last commit.


---

## 1. Setup and Configuration

First-time Git setup. You only need to do this once per computer.

In [ ]:
%%bash
# Tell Git who you are (used in commit metadata)
git config --global user.name "Your Name"
git config --global user.email "your.email@university.edu"

# Set the default branch name to 'main' (modern convention)
git config --global init.defaultBranch main

# Set your preferred text editor for commit messages
git config --global core.editor "nano"    # or "vim", or "code --wait" for VS Code

# Verify your configuration
git config --list

---

## 2. Core Concepts: The Three States

Every file in a Git repo exists in one of three states. Understanding this model
is the key to understanding everything else in Git.

```
+----------------+     git add     +----------------+    git commit    +----------------+
|    WORKING     | -------------> |    STAGING      | -------------> |   REPOSITORY   |
|   DIRECTORY    |                |   AREA (index)  |                |   (commits)    |
+----------------+                +----------------+                +----------------+

  Your files as         Files marked           Permanent snapshots
  you edit them         "ready to commit"      of your project
```

**Why a staging area?** It lets you commit a subset of your changes. For example,
you fixed a bug AND started a new feature in the same session. You can stage and
commit the bug fix first, then commit the new feature separately. This keeps your
history clean and each commit focused.

---

## 3. Creating a Repository

Two ways to start: initialize a new one, or clone an existing one.

In [ ]:
%%bash
# Option 1: Start a new repository from scratch
mkdir my_rna_seq_analysis
cd my_rna_seq_analysis
git init
# This creates a hidden .git/ directory that stores all version history

# Option 2: Clone an existing repository from GitHub
# git clone https://github.com/username/repository.git
# git clone https://github.com/username/repository.git my_local_name

# Check the status of your repo
git status

### Setting Up a Bioinformatics Project with Git

Let's create a real project structure and initialize it properly.

In [ ]:
%%bash
# Create project structure
mkdir -p /tmp/gene_expression_study/{data,scripts,results,docs}
cd /tmp/gene_expression_study
git init

# Create a README
cat > README.md << 'EOF'
# Gene Expression Study

Analysis of differential gene expression in breast cancer subtypes.

## Directory Structure
- `data/` - Raw and processed data (not tracked by Git)
- `scripts/` - Analysis scripts and pipelines
- `results/` - Generated results (not tracked by Git)
- `docs/` - Documentation and notes
EOF

echo "Project created and initialized!"
git status

---

## 4. The .gitignore File

In bioinformatics, most files should NOT be in Git. Sequence data, alignment files,
and results are too large (Git has a ~100 MB file limit on GitHub) and can be
regenerated from scripts + raw data.

**Rule of thumb: Track code and configuration. Do NOT track data and outputs.**

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Create a comprehensive .gitignore for bioinformatics
cat > .gitignore << 'GITIGNORE'
# ==========================================
# Bioinformatics .gitignore
# ==========================================

# --- Large data files ---
*.fastq
*.fastq.gz
*.fq.gz
*.bam
*.bam.bai
*.sam
*.cram
*.bcf
*.sra
data/raw/

# --- Reference genomes ---
*.fa
*.fasta
*.fa.fai
*.dict

# --- Results and outputs (regenerated by scripts) ---
results/
*.log
*.tmp
*.out

# --- Python ---
__pycache__/
*.pyc
.ipynb_checkpoints/
*.egg-info/

# --- R ---
.Rhistory
.RData

# --- OS files ---
.DS_Store
Thumbs.db

# --- IDE files ---
.vscode/
.idea/
GITIGNORE

echo ".gitignore created"
cat .gitignore

**What TO track in Git:**

| Track | Do NOT Track |
|-------|-------------|
| Analysis scripts (`.py`, `.R`, `.sh`) | Raw data (`.fastq.gz`, `.bam`) |
| Pipeline definitions (Snakemake, Nextflow) | Reference genomes |
| Configuration files | Generated results |
| Documentation (README, methods) | Log files |
| Environment specs (`environment.yml`) | OS/IDE metadata |
| Small metadata files (sample sheets) | Anything >50 MB |

---

## 5. The Basic Workflow: add, commit, push

This is the workflow you will use dozens of times per day.

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Step 1: Check what has changed
git status

# Step 2: Stage files for commit
git add README.md               # Stage a specific file
git add .gitignore              # Stage another file
# git add .                     # Stage ALL changes (use with caution)
# git add scripts/              # Stage all files in a directory

# Step 3: Commit with a descriptive message
git commit -m "Initialize project with README and .gitignore"

# Verify
git log --oneline

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Let's add an analysis script and commit it
cat > scripts/deseq2_analysis.R << 'EOF'
# DESeq2 Differential Expression Analysis
# Author: Your Name
# Date: 2024-01-15

library(DESeq2)

# Load count matrix
counts <- read.csv("data/counts_matrix.csv", row.names=1)
metadata <- read.csv("data/sample_metadata.csv")

# Create DESeq2 object
dds <- DESeqDataSetFromMatrix(countData = counts,
                              colData = metadata,
                              design = ~ condition)

# Run differential expression
dds <- DESeq(dds)
results <- results(dds, alpha = 0.05)

# Save results
write.csv(as.data.frame(results), "results/deseq2_results.csv")
EOF

git add scripts/deseq2_analysis.R
git commit -m "Add DESeq2 differential expression analysis script"
git log --oneline

### Writing Good Commit Messages

Commit messages are for your future self and your collaborators. They should explain
the *why*, not just the *what*.

```
BAD commit messages:             GOOD commit messages:

  "fix"                            "Fix off-by-one error in codon extraction"
  "update"                         "Add support for GFF3 annotation format"
  "stuff"                          "Optimize GC content calc (2x faster)"
  "final version"                  "Complete DESeq2 analysis with batch correction"
  "asdf"                           "Filter low-quality variants (QUAL < 20)"

Template:  <verb> <what> [context/reason]

Common verbs: Add, Fix, Update, Remove, Refactor, Optimize, Implement
```

For longer messages, use the editor instead of `-m`:

```bash
git commit    # Opens your editor for a multi-line message
```

Multi-line message convention:
```
Short summary (50 chars or less)

Detailed explanation of what changed and why. Wrap at 72 characters.
Reference relevant issues: Fixes #42
```

---

## 6. Viewing History and Differences

Git's real power is in navigating history -- comparing versions, finding when a bug
was introduced, or recovering old code.

In [ ]:
%%bash
cd /tmp/gene_expression_study

# View commit history
git log                          # Full history (press q to exit)
git log --oneline                # Compact: one line per commit
git log --oneline -5             # Last 5 commits
git log --graph --oneline        # Visual branch history
git log --stat                   # Show which files changed per commit

# View changes
git diff                         # Unstaged changes (working dir vs staging)
git diff --staged                # Staged changes (staging vs last commit)
git diff HEAD~1                  # Changes since the previous commit
# git diff abc123 def456         # Changes between two specific commits

# Show details of a specific commit
git show HEAD                    # Latest commit with full diff
# git show abc123                # Specific commit by hash

### Practical example: Tracking pipeline changes

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Modify the analysis script -- change the significance threshold
sed -i '' 's/alpha = 0.05/alpha = 0.01/' scripts/deseq2_analysis.R 2>/dev/null ||
sed -i 's/alpha = 0.05/alpha = 0.01/' scripts/deseq2_analysis.R

# See what changed
echo "=== What changed? ==="
git diff scripts/deseq2_analysis.R

# Commit the change
git add scripts/deseq2_analysis.R
git commit -m "Tighten significance threshold from 0.05 to 0.01"

echo ""
echo "=== Commit history ==="
git log --oneline

---

## 7. Undoing Mistakes

Everyone makes mistakes. Git gives you multiple safety nets, from gentle to aggressive.

In [ ]:
%%bash
# === DISCARD CHANGES in working directory ===
# (File is modified but not yet staged)
# git restore filename.py               # Modern syntax (Git 2.23+)
# git checkout -- filename.py           # Older syntax

# === UNSTAGE a file ===
# (File is staged but not yet committed)
# git restore --staged filename.py      # Modern syntax
# git reset HEAD filename.py            # Older syntax

# === UNDO the last commit (keep changes) ===
# git reset --soft HEAD~1               # Changes remain staged
# git reset HEAD~1                      # Changes become unstaged

# === UNDO the last commit (DISCARD changes) ===
# git reset --hard HEAD~1               # DANGEROUS: changes are lost!

# === SAFELY undo a specific commit ===
# (Creates a NEW commit that reverses the old one -- safe for shared repos)
# git revert abc123

echo 'Summary of undo operations:'
echo '  restore file      = discard local changes'
echo '  restore --staged  = unstage'
echo '  reset --soft      = undo commit, keep changes staged'
echo '  reset --hard      = undo commit, DELETE changes (dangerous)'
echo '  revert            = new commit that undoes an old one (safe)'

### When to use each undo method

| Situation | Command | Safe? |
|-----------|---------|-------|
| Accidentally edited a file, want to restore it | `git restore file` | Yes (loses edits) |
| Staged wrong file, want to unstage | `git restore --staged file` | Yes |
| Just committed, forgot a file | `git reset --soft HEAD~1` | Yes |
| Just committed, want to discard everything | `git reset --hard HEAD~1` | DANGEROUS |
| Need to undo an old commit in shared repo | `git revert <hash>` | Yes |

---

## 8. Branches

Branches let you work on different things simultaneously without interfering with
each other. This is essential when:

- You want to try a different normalization method without breaking the working pipeline
- Multiple lab members are working on different analyses
- You need to fix a bug while a new feature is half-done

```
main -------*--------*---------------------*---*---> (stable code)
             \                             /   /
              *---*---*---*--- feature ---/   /      (new analysis method)
                   \                        /
                    *---*--- bugfix -------/         (quick fix)
```

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Create a new branch and switch to it
git checkout -b feature-batch-correction
# Modern alternative: git switch -c feature-batch-correction

# Make changes on the branch
cat > scripts/batch_correction.R << 'EOF'
# Batch correction using ComBat-seq
library(sva)

# Load data
counts <- read.csv("data/counts_matrix.csv", row.names=1)
batch <- read.csv("data/batch_info.csv")$batch

# Apply ComBat-seq
corrected <- ComBat_seq(counts, batch=batch)
write.csv(corrected, "data/counts_batch_corrected.csv")
EOF

git add scripts/batch_correction.R
git commit -m "Add batch correction with ComBat-seq"

echo "=== Current branch ==="
git branch

echo ""
echo "=== Log on this branch ==="
git log --oneline

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Switch back to main and merge the feature branch
git checkout main
git merge feature-batch-correction

# Clean up: delete the merged branch
git branch -d feature-batch-correction

echo "=== Branches after merge ==="
git branch

echo ""
echo "=== Full history ==="
git log --oneline --graph

### Branch commands reference

```bash
git branch                        # List local branches
git branch -a                     # List all branches (including remote)
git branch feature-name           # Create a new branch
git checkout feature-name         # Switch to a branch
git checkout -b feature-name      # Create AND switch (shortcut)
git switch feature-name           # Modern switch (Git 2.23+)
git switch -c feature-name        # Modern create + switch
git merge feature-name            # Merge branch into current branch
git branch -d feature-name        # Delete branch (after merging)
git branch -D feature-name        # Force-delete branch (even if unmerged)
```

### Branch naming conventions

```
feature-batch-correction     # New feature
bugfix-gc-calculation        # Bug fix
experiment-limma-vs-deseq2   # Experimental analysis (might be discarded)
refactor-pipeline-structure  # Code reorganization
```

---

## 9. Working with Remote Repositories (GitHub)

GitHub (or GitLab, Bitbucket) hosts your repository online. This enables:
- Backup (your laptop could die)
- Collaboration (lab members can contribute)
- Publication (share code with reviewers and readers)
- Visibility (potential employers can see your work)

In [ ]:
%%bash
# Link your local repo to GitHub
# (First, create an empty repo on github.com, then:)
# git remote add origin https://github.com/username/gene_expression_study.git

# Verify the remote
# git remote -v

# Push your code to GitHub
# git push -u origin main           # First push: -u sets upstream tracking
# git push                           # Subsequent pushes: just this

# Get changes from GitHub
# git fetch                          # Download changes (do not merge)
# git pull                           # Fetch + merge in one step
# git pull --rebase                  # Fetch + rebase (cleaner history)

echo 'Remote operations -- requires a GitHub account and repo'

### The Daily Collaboration Workflow

```
1. START OF DAY
   git pull                        # Get any changes your collaborators pushed

2. DO YOUR WORK
   ... edit files ...

3. SAVE YOUR PROGRESS (multiple times per day)
   git status                      # Review what changed
   git add <files>                 # Stage changes
   git commit -m "message"         # Commit

4. END OF DAY
   git push                        # Share your work
```

### Pull Requests

When collaborating on GitHub, changes are proposed via **Pull Requests** (PRs):

1. Create a branch: `git checkout -b feature-new-analysis`
2. Make your changes and commit
3. Push the branch: `git push -u origin feature-new-analysis`
4. On GitHub, open a Pull Request from your branch to `main`
5. Collaborators review your code, leave comments
6. After approval, merge the PR on GitHub
7. Locally: `git checkout main && git pull`

---

## 10. Handling Merge Conflicts

Conflicts happen when two people edit the same lines in the same file. Git cannot
automatically decide which version to keep, so you must resolve it manually.

```
When you see:

<<<<<<< HEAD
alpha = 0.01    # Your version
=======
alpha = 0.05    # Their version
>>>>>>> feature-branch

You must:
1. Choose which version to keep (or combine them)
2. Delete the conflict markers (<<<, ===, >>>)
3. Stage and commit the resolved file
```

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Simulate a merge conflict
git checkout -b experiment-threshold
sed -i '' 's/alpha = 0.01/alpha = 0.001/' scripts/deseq2_analysis.R 2>/dev/null ||
sed -i 's/alpha = 0.01/alpha = 0.001/' scripts/deseq2_analysis.R
git add scripts/deseq2_analysis.R
git commit -m "Use stricter threshold for experiments"

git checkout main
sed -i '' 's/alpha = 0.01/alpha = 0.05/' scripts/deseq2_analysis.R 2>/dev/null ||
sed -i 's/alpha = 0.01/alpha = 0.05/' scripts/deseq2_analysis.R
git add scripts/deseq2_analysis.R
git commit -m "Revert to standard threshold"

# Try to merge -- this will conflict
echo "=== Attempting merge ==="
git merge experiment-threshold || echo "Conflict! Need to resolve manually."

echo ""
echo "=== Conflicted file ==="
cat scripts/deseq2_analysis.R | head -15

In [ ]:
%%bash
cd /tmp/gene_expression_study

# Resolve the conflict -- let's keep the 0.05 threshold
# In real life, you would edit the file in your editor.
# Here we use sed to clean it up:
sed -i '' '/<<<<<<</d; /=======/d; />>>>>>>/d; /alpha = 0.001/d' scripts/deseq2_analysis.R 2>/dev/null ||
sed -i '/<<<<<<</d; /=======/d; />>>>>>>/d; /alpha = 0.001/d' scripts/deseq2_analysis.R

git add scripts/deseq2_analysis.R
git commit -m "Resolve merge conflict: keep standard 0.05 threshold"
git branch -d experiment-threshold

echo "Conflict resolved!"
git log --oneline

---

## 11. Advanced: git stash

Sometimes you need to switch branches but you have uncommitted changes that you
do not want to commit yet. `git stash` temporarily shelves your changes.

```bash
# Save current changes to the stash
git stash                        # Stash with automatic message
git stash save "WIP: new plot"   # Stash with custom message

# Now you can safely switch branches, do other work, etc.
git checkout other-branch
# ... do stuff ...
git checkout main

# Restore your stashed changes
git stash pop                    # Restore and remove from stash
git stash apply                  # Restore but keep in stash

# List all stashes
git stash list
```

---

## 12. Advanced: Tags for Releases

Tags mark specific commits -- useful for marking paper submissions, tool releases,
or stable pipeline versions.

```bash
# Create a lightweight tag
git tag v1.0

# Create an annotated tag (preferred -- includes message and metadata)
git tag -a v1.0 -m "Pipeline version used for Smith et al. 2024 paper"

# List tags
git tag

# Push tags to GitHub
git push --tags

# Checkout a specific tag (read-only, for inspection)
git checkout v1.0
```

**Bioinformatics use case:** When you submit a paper, tag the exact version of
your analysis code. If a reviewer asks a question, you can always go back to
exactly the code that produced the results.

---

## 13. Best Practices for Scientific Code

1. **Commit often**: Each commit should represent one logical change
2. **Write clear messages**: "Fix bug" is useless; "Fix off-by-one error in exon boundary parsing" is useful
3. **Never commit large data files**: Use `.gitignore` aggressively
4. **Use branches for experiments**: Do not experiment on main
5. **Tag paper submissions**: `git tag -a v1.0 -m "Submitted to Nature Genetics"`
6. **Include an environment file**: `conda env export > environment.yml`
7. **Pull before push**: Avoid unnecessary merge conflicts
8. **Make your repo public when you publish**: Reviewers and readers should be able to reproduce your work
9. **Add a README**: Explain what the project does and how to run it
10. **Add a LICENSE**: Without one, your code is technically unusable by others (MIT or BSD-3 are good defaults for academic code)

---

## Quick Reference Card

```
SETUP                           DAILY WORKFLOW
  git init                        git status
  git clone URL                   git add file
  git config --global ...         git commit -m "message"
                                  git push / git pull

HISTORY                         BRANCHES
  git log --oneline               git branch
  git diff                        git checkout -b name
  git show HEAD                   git merge name
  git log --graph                 git branch -d name

UNDO                            REMOTE
  git restore file                git remote add origin URL
  git restore --staged file       git push -u origin main
  git reset --soft HEAD~1         git pull
  git revert <hash>               git fetch
```

---

## Exercises

These exercises walk you through realistic Git workflows in bioinformatics.

### Exercise 1: Initialize a Bioinformatics Project

1. Create a new directory called `variant_calling_pipeline`
2. Initialize a Git repository
3. Create a proper `.gitignore` that excludes BAM, VCF, FASTQ, and log files
4. Create a `README.md` explaining the project
5. Make an initial commit with both files

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 2: Feature Branch Workflow

Using the repo from Exercise 1:
1. Create a branch called `feature-quality-filter`
2. Create a script `scripts/filter_variants.sh` that uses `bcftools filter`
3. Commit the script
4. Switch back to main and merge the feature branch
5. Delete the feature branch

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 3: Undo Practice

1. Modify the README.md (add a line)
2. Stage it with `git add`
3. Unstage it with `git restore --staged`
4. Discard the changes with `git restore`
5. Verify the file is back to its original state

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 4: Investigate History

Using the gene_expression_study repo we built earlier:
1. View the full commit log in one-line format
2. Show the diff of the second commit (the one that added the DESeq2 script)
3. Find which commit changed the alpha threshold
4. Show the state of `scripts/deseq2_analysis.R` at the first commit

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 5: Publish to GitHub

This is a walkthrough (not executable in Jupyter) to practice later:

1. Create a new repository on github.com (do NOT initialize with README)
2. Link your local repo to it
3. Push your code
4. Make a change locally, commit, and push again
5. Verify on github.com that your changes appear

Write the commands you would use:

In [ ]:
%%bash
# YOUR SOLUTION (write the commands, even if you cannot run them now):


---

## Exercise Solutions

### Solution 1

In [ ]:
%%bash
mkdir -p /tmp/variant_calling_pipeline/scripts
cd /tmp/variant_calling_pipeline
git init

cat > .gitignore << 'EOF'
*.bam
*.bam.bai
*.sam
*.vcf
*.vcf.gz
*.fastq
*.fastq.gz
*.log
*.tmp
results/
__pycache__/
.DS_Store
EOF

cat > README.md << 'EOF'
# Variant Calling Pipeline

GATK-based germline variant calling pipeline for whole-genome sequencing data.

## Steps
1. Read alignment with BWA-MEM
2. Duplicate marking with Picard
3. Base quality score recalibration (BQSR)
4. Variant calling with HaplotypeCaller
5. Variant filtering (VQSR)
EOF

git add README.md .gitignore
git commit -m "Initialize variant calling pipeline project"
git log --oneline

### Solution 2

In [ ]:
%%bash
cd /tmp/variant_calling_pipeline

git checkout -b feature-quality-filter

cat > scripts/filter_variants.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

INPUT_VCF=$1
OUTPUT_VCF=$2

bcftools filter \
    -i 'QUAL>30 && DP>10 && MQ>40' \
    -o "$OUTPUT_VCF" \
    "$INPUT_VCF"

echo "Filtered variants saved to $OUTPUT_VCF"
SCRIPT

chmod +x scripts/filter_variants.sh
git add scripts/filter_variants.sh
git commit -m "Add variant quality filter script using bcftools"

git checkout main
git merge feature-quality-filter
git branch -d feature-quality-filter

git log --oneline --graph

### Solution 3

In [ ]:
%%bash
cd /tmp/variant_calling_pipeline

# 1. Modify README
echo "## Authors" >> README.md
echo "Your Name" >> README.md

# 2. Stage it
git add README.md
echo "=== After staging ==="
git status

# 3. Unstage it
git restore --staged README.md
echo ""
echo "=== After unstaging ==="
git status

# 4. Discard changes
git restore README.md
echo ""
echo "=== After restoring ==="
git status

# 5. Verify
echo ""
echo "=== README content (should NOT have Authors section) ==="
cat README.md

### Solution 4

In [ ]:
%%bash
cd /tmp/gene_expression_study

# 1. Full commit log
echo "=== Commit log ==="
git log --oneline

# 2. Show the diff of the second commit
echo ""
echo "=== Second commit diff ==="
second_commit=$(git log --oneline | tail -2 | head -1 | awk '{print $1}')
git show "$second_commit" --stat

# 3. Find which commit changed alpha
echo ""
echo "=== Commits that changed alpha ==="
git log --oneline --all -S "alpha" 

# 4. Show deseq2_analysis.R at the first commit
echo ""
echo "=== DESeq2 script at first relevant commit ==="
first_with_deseq=$(git log --oneline -- scripts/deseq2_analysis.R | tail -1 | awk '{print $1}')
git show "${first_with_deseq}:scripts/deseq2_analysis.R" 2>/dev/null | head -10 || echo "(file did not exist in first commit)"

### Solution 5

```bash
# 1. On github.com: Click "New Repository", name it, do NOT add README

# 2. Link local repo to GitHub
git remote add origin https://github.com/yourusername/variant_calling_pipeline.git

# 3. Push code
git push -u origin main

# 4. Make a change, commit, push
echo "## License" >> README.md
echo "MIT" >> README.md
git add README.md
git commit -m "Add MIT license to README"
git push

# 5. Go to github.com/yourusername/variant_calling_pipeline to verify
```

---

## Key Takeaways

1. **Commit often**: Small, logical commits with descriptive messages
2. **Use .gitignore aggressively**: Never commit large data files
3. **Branches for experiments**: Keep main stable, experiment on branches
4. **Pull before push**: Minimize merge conflicts
5. **Tag your paper submissions**: `git tag -a v1.0 -m "Submitted to journal"`
6. **Make repos public when you publish**: Reproducibility requires code access

---

### Next Module: Bash Scripting -->

---

[< Previous: Module 0.1: Linux Fundamentals for Bioinformatics](../01_Linux_Basics/01_linux_fundamentals.ipynb) | [Tier 0: Computational Foundations Overview](../README.md) | [Next: Module 0.3: Bash Scripting for Bioinformatics >](../03_Bash_Scripting/01_bash_scripting.ipynb)

---